In [ ]:
# ============================================
# 0. 安装依赖（Colab）
# ============================================
!pip -q install pandas numpy requests folium tqdm

# ============================================
# 1. 导入库
# ============================================
import time
import requests
import pandas as pd
import numpy as np
import folium

from tqdm import tqdm
from folium import FeatureGroup, LayerControl
from folium.plugins import Fullscreen
from IPython.display import display

# ============================================
# 2. Google Places API (New)
#    先去 Google Cloud 开启 Places API (New)
#    并填写你的 API KEY
# ============================================
GOOGLE_API_KEY = "YOUR_GOOGLE_MAPS_API_KEY"
PLACES_TEXT_SEARCH_URL = "https://places.googleapis.com/v1/places:searchText"

if GOOGLE_API_KEY == "YOUR_GOOGLE_MAPS_API_KEY":
    raise ValueError("请先把 GOOGLE_API_KEY 改成你自己的 Google Maps API Key。")

# ============================================
# 3. 威尼斯主岛 bbox
#    可自行微调
# ============================================
west, south, east, north = 12.2960, 45.4280, 12.3710, 45.4555

# ============================================
# 4. 搜索配置
#    Text Search (New) 每页最多 20 条，总计最多 60 条/查询
#    所以这里通过多组中英意查询来尽量覆盖更多地点
# ============================================
PAGE_SIZE = 20
SLEEP_SECONDS = 0.4

# 颜色配置
CATEGORY_COLORS = {
    "attraction": "#6ec1ff",
    "hotel": "#ffffff",
    "shop": "#ffd166",
    "restaurant": "#ff6b6b"
}

# ============================================
# 5. 类别查询词（英文 + 意大利语）
#    attraction 扩展到：
#    tourist attraction / church / piazza / photo spot 等
# ============================================
CATEGORY_QUERIES = {
    "restaurant": [
        {"textQuery": "restaurant in Venice", "includedType": "restaurant"},
        {"textQuery": "ristorante a Venezia", "includedType": "restaurant"},
        {"textQuery": "trattoria a Venezia", "includedType": "restaurant"},
        {"textQuery": "osteria a Venezia", "includedType": "restaurant"},
    ],
    "hotel": [
        {"textQuery": "hotel in Venice", "includedType": "lodging"},
        {"textQuery": "albergo a Venezia", "includedType": "lodging"},
        {"textQuery": "hotel a Venezia", "includedType": "lodging"},
    ],
    "shop": [
        {"textQuery": "shop in Venice", "includedType": "store"},
        {"textQuery": "store in Venice", "includedType": "store"},
        {"textQuery": "negozio a Venezia", "includedType": "store"},
        {"textQuery": "boutique a Venezia", "includedType": "store"},
    ],
    "attraction": [
        {"textQuery": "tourist attraction in Venice", "includedType": "tourist_attraction"},
        {"textQuery": "attrazione turistica a Venezia", "includedType": "tourist_attraction"},
        {"textQuery": "church in Venice", "includedType": "church"},
        {"textQuery": "chiesa a Venezia", "includedType": "church"},
        {"textQuery": "piazza in Venice", "includedType": None},
        {"textQuery": "piazza a Venezia", "includedType": None},
        {"textQuery": "photo spot in Venice", "includedType": None},
        {"textQuery": "photography spot in Venice", "includedType": None},
        {"textQuery": "viewpoint in Venice", "includedType": None},
        {"textQuery": "belvedere a Venezia", "includedType": None},
        {"textQuery": "museum in Venice", "includedType": "museum"},
        {"textQuery": "museo a Venezia", "includedType": "museum"},
        {"textQuery": "historical landmark in Venice", "includedType": None},
        {"textQuery": "luogo storico a Venezia", "includedType": None},
    ]
}

# ============================================
# 6. Google Places Text Search (New)
# ============================================
def google_text_search(
    api_key: str,
    text_query: str,
    included_type: str = None,
    page_token: str = None,
    page_size: int = 20
):
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": api_key,
        # 只取需要字段，减少成本
        "X-Goog-FieldMask": ",".join([
            "places.id",
            "places.displayName",
            "places.formattedAddress",
            "places.location",
            "places.primaryType",
            "places.types",
            "nextPageToken"
        ])
    }

    body = {
        "textQuery": text_query,
        "pageSize": page_size,
        "locationRestriction": {
            "rectangle": {
                "low": {
                    "latitude": south,
                    "longitude": west
                },
                "high": {
                    "latitude": north,
                    "longitude": east
                }
            }
        }
    }

    if included_type:
        body["includedType"] = included_type
        body["strictTypeFiltering"] = False

    if page_token:
        body["pageToken"] = page_token

    r = requests.post(
        PLACES_TEXT_SEARCH_URL,
        headers=headers,
        json=body,
        timeout=60
    )
    r.raise_for_status()
    return r.json()

# ============================================
# 7. 抓取所有类别
#    注意：
#    Text Search 单查询最多 60 条，所以通过多查询并去重
# ============================================
all_rows = []
seen_place_ids = set()

for category, queries in CATEGORY_QUERIES.items():
    print(f"\n开始抓取类别：{category}")

    for q in tqdm(queries, desc=f"{category} queries"):
        text_query = q["textQuery"]
        included_type = q["includedType"]

        next_page_token = None
        page_no = 0

        while True:
            try:
                data = google_text_search(
                    api_key=GOOGLE_API_KEY,
                    text_query=text_query,
                    included_type=included_type,
                    page_token=next_page_token,
                    page_size=PAGE_SIZE
                )

                places = data.get("places", [])
                if not places:
                    break

                for p in places:
                    place_id = p.get("id")
                    if not place_id or place_id in seen_place_ids:
                        continue
                    seen_place_ids.add(place_id)

                    display_name = (p.get("displayName") or {}).get("text", "")
                    loc = p.get("location", {})
                    lat = loc.get("latitude", np.nan)
                    lon = loc.get("longitude", np.nan)

                    row = {
                        "place_id": place_id,
                        "name": display_name,
                        "category": category,
                        "lat": pd.to_numeric(lat, errors="coerce"),
                        "lon": pd.to_numeric(lon, errors="coerce"),
                        "address": p.get("formattedAddress", ""),
                        "primary_type": p.get("primaryType", ""),
                        "types": ", ".join(p.get("types", [])) if p.get("types") else "",
                        "query_used": text_query,
                        "source": "Google Maps"
                    }
                    all_rows.append(row)

                next_page_token = data.get("nextPageToken")
                page_no += 1

                # Text Search 单查询最多 60 结果，也就是最多 3 页
                if not next_page_token or page_no >= 3:
                    break

                time.sleep(SLEEP_SECONDS)

            except Exception as e:
                print(f"[警告] {category} / {text_query} 抓取失败：{e}")
                break

df = pd.DataFrame(all_rows)

# ============================================
# 8. 坐标清洗 + bbox 再过滤
# ============================================
df = df.dropna(subset=["lat", "lon"]).copy()

df = df[
    (df["lon"] >= west) & (df["lon"] <= east) &
    (df["lat"] >= south) & (df["lat"] <= north)
].copy()

# 名称为空的去掉
df = df[df["name"].astype(str).str.strip() != ""].copy()

# 再次按类别+名称+坐标轻度去重
df = (
    df.sort_values(["category", "name"])
      .drop_duplicates(subset=["category", "name", "lat", "lon"])
      .reset_index(drop=True)
)

print(f"\n最终点位总数：{len(df)}")
print(df["category"].value_counts(dropna=False))
display(df.head(10))

# ============================================
# 9. 导出 CSV
#    只保留你要的字段：类别、名称、经纬度
# ============================================
csv_df = (
    df[["category", "name", "lat", "lon"]]
    .rename(columns={
        "category": "类型",
        "name": "名称",
        "lat": "纬度",
        "lon": "经度"
    })
    .sort_values(["类型", "名称"])
    .reset_index(drop=True)
)

csv_path = "/content/venice_google_places_points.csv"
csv_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print(f"\nCSV 已导出：{csv_path}")
display(csv_df.head(20))

# ============================================
# 10. 画在线地图
# ============================================
if len(df) == 0:
    raise ValueError("没有抓到可绘图的数据。请检查 API Key、billing、bbox 或查询词。")

center_lat = df["lat"].mean()
center_lon = df["lon"].mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=15,
    tiles="CartoDB dark_matter",
    control_scale=True
)

Fullscreen().add_to(m)

# 画 bbox 白线
bbox_coords = [
    [south, west],
    [south, east],
    [north, east],
    [north, west],
    [south, west]
]

folium.PolyLine(
    locations=bbox_coords,
    color="white",
    weight=2.5,
    opacity=1
).add_to(m)

# 图层
fg_attraction = FeatureGroup(name="Attraction", show=True)
fg_hotel = FeatureGroup(name="Hotel", show=True)
fg_shop = FeatureGroup(name="Shop", show=True)
fg_restaurant = FeatureGroup(name="Restaurant", show=True)

layer_map = {
    "attraction": fg_attraction,
    "hotel": fg_hotel,
    "shop": fg_shop,
    "restaurant": fg_restaurant
}

for _, row in df.iterrows():
    category = row["category"]
    name = row["name"]
    lat = row["lat"]
    lon = row["lon"]
    address = row["address"]
    primary_type = row["primary_type"]
    types = row["types"]
    query_used = row["query_used"]

    color = CATEGORY_COLORS.get(category, "#cccccc")
    radius = 6 if category == "hotel" else 5

    popup_html = f"""
    <b>{name}</b><br>
    Category: {category}<br>
    Lat: {lat:.6f}<br>
    Lon: {lon:.6f}<br>
    Address: {address}<br>
    Primary type: {primary_type}<br>
    Types: {types}<br>
    Query used: {query_used}
    """

    folium.CircleMarker(
        location=(lat, lon),
        radius=radius,
        color=color,
        weight=1.5,
        fill=True,
        fill_color=color,
        fill_opacity=0.95,
        popup=popup_html
    ).add_to(layer_map[category])

fg_attraction.add_to(m)
fg_hotel.add_to(m)
fg_shop.add_to(m)
fg_restaurant.add_to(m)

LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[south, west], [north, east]])

legend_html = """
<div style="
    position: fixed;
    bottom: 30px;
    left: 30px;
    z-index: 9999;
    background-color: rgba(0,0,0,0.80);
    color: white;
    padding: 12px 14px;
    border-radius: 8px;
    font-size: 14px;
    line-height: 1.6;
    box-shadow: 0 0 8px rgba(255,255,255,0.15);
">
    <div style="font-weight: bold; margin-bottom: 6px;">Venice Main Island - Google Places</div>
    <div><span style="color:#6ec1ff;">●</span> Attraction</div>
    <div><span style="color:#ffffff;">●</span> Hotel</div>
    <div><span style="color:#ffd166;">●</span> Shop</div>
    <div><span style="color:#ff6b6b;">●</span> Restaurant</div>
    <div><span style="color:#ffffff;">━━</span> Venice bbox</div>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

print("\n地图已生成，下面直接在线显示。")
display(m)